In [0]:
import json
from pathlib import Path
import pandas as pd
from pyspark.sql.functions import to_date



In [0]:
%sql
DROP DATABASE IF EXISTS workspace.silver CASCADE; 

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados'

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.silver.predios_registro")

In [0]:

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.predios_registro (
    pk STRING,
    matricula STRING,
    fecha_radica_texto STRING,
    fecha_apertura_texto STRING,
    year_radica STRING,
    orip STRING,
    divipola STRING,
    departamento STRING,
    municipio STRING,
    tipo_predio_zona STRING,
    categoria_ruralidad_2024 STRING,
    num_anotacion INT,
    estado_folio STRING,
    folios_derivados STRING,
    dinamica_2024 INT,
    cod_natujur INT,
    nombre_natujur STRING,
    documento_justificativo STRING,
    count_a INT,
    count_de INT,
    predios_nuevos INT,
    tiene_valor INT,
    tiene_mas_de_un_valor INT,
    valor DOUBLE
)
""")


In [0]:
# Limpia nombres de columnas para estandarizarlos:
# - quita espacios al inicio/fin
# - convierte a minúsculas
# - reemplaza espacios, puntos y guiones por "_"
def clean_column_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_")

# Convierte valores complejos a formatos comparables:
# - list  -> tuple
# - ndarray -> tuple
# - dict -> tuple ordenada de pares (key, value)
def normalize_cell(value):
    if isinstance(value, list):
        return tuple(value)
    if isinstance(value, np.ndarray):
        return tuple(value.tolist())
    if isinstance(value, dict):
        return tuple(sorted(value.items()))
    return value

# Si el valor es lista/tupla, lo convierte a string separado por comas.
# Si no, lo deja igual.
# Se usa para columnas como géneros o días de programación.
def join_if_sequence(value):
    if isinstance(value, (list, tuple)):
        return ",".join(str(item) for item in value)
    return value

#Limpiar fechas
#Para fecha radica 07/08/2022 -> 07/08/2022
#Para fecha apertura texto 2022-08-07 00:00:00 -> 07/08/2022
def clean_fecha(value):
    from datetime import datetime
    
    if not value:
        return None
    
    value = value.strip()
    
    formatos = (
        "%d/%m/%y",          # fecha_radica
        "%d/%m/%Y",         # fecha_radica
        "%Y-%m-%d %H:%M:%S" # fecha_apertura
    )
    
    for fmt in formatos:
        try:
            fecha = datetime.strptime(value, fmt)
            return fecha.strftime("%d/%m/%Y")  # formato final requerido
        except:
            continue
    
    return None

#Funcion para tipar el campo orip a tres dígitos 2 -> 002
def clean_orip(value):
    if not value:
        return None
    return str(value).zfill(3)

#Función para tipar el campo davipola a 5 digitos 5002 -> 05002
def clean_divipola(value):
    if not value:
        return None
    return str(value).zfill(5)

#Función para convertir a mayúsculas tipo_predio Rural -> RURAL
def clean_categoria(value):
    if not value or str(value).strip() == "":
        return None
    return str(value).strip().upper()

In [0]:
# Leer bronze y pasar a pandas
df_spark = spark.table("workspace.bronze.predios_registro")
df = df_spark.toPandas()

In [0]:
# Limpiar nombres de columnas
df.columns = [clean_column_name(col) for col in df.columns]

In [0]:
# Renombrar columnas clave
rename_map = {
    "pk": "transaction_id",
    "matricula": "registry_id",
    "fecha_radica_texto" : "filing_date",
    "year_radica": "filing_year",
    "orip": "orip",
    "divipola": "divipola",
    "departamento": "department",
    "municipio": "municipality",
    "tipo_predio_zona": "property_zone_type",
    "categoria_ruralidad": "categoria_ruralidad",
    "num_anotacion": "annotation_number",
    "estado_folio": "registry_status",
    "folios_derivados": "derived_folios",
    "dinamica": "dynamic",
    "cod_natujur": "nature_code",
    "nombre_natujur": "nature_name",
    "numero_catastral": "cadastral_id",
    "numero_catastral_antiguo": "old_cadastral_id",
    "documento_justificativo": "supporting_document",
    "count_a": "receivers_count",
    "count_de": "grantors_count",
    "predios_nuevos": "new_property",
    "tiene_valor": "has_value",
    "tiene_mas_de_un_valor": "has_multiple_values",
    "valor": "transaction_value"
    
}

df = df.rename(columns=rename_map, errors="ignore")
print(df.columns)

In [0]:
# 6) Quitar duplicados
df = df.drop_duplicates()

In [0]:
# 8) Eliminar columnas duplicadas (si las hubiera)
df = df.loc[:, ~df.columns.duplicated()]

# 9) Regresar a Spark DataFrame
df_spark = spark.createDataFrame(df)


In [0]:
# Normalizar fecha_radica_texto en pandas antes de escribir
import pandas as pd

# Convertir fecha_radica_texto a datetime usando pandas (infiere formatos automáticamente)
df['filing_date'] = pd.to_datetime(df['filing_date'], format='mixed', errors='coerce')

# Convertir de nuevo a Spark
df_spark = spark.createDataFrame(df)

# Escribir a Delta
df_spark.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.predios_registro")

In [0]:
# Ejemplo de selección de columna existente
spark.sql("""
SELECT pk
FROM silver.predios_registro
LIMIT 5
""").show()